## Download ERA5-Land Datasets

This section will guide you through downloading ERA5-Land data from the Copernicus Climate Data Store (CDS).

**Important Considerations:**

1.  **CDS API Credentials**: You need a registered account on the Copernicus Climate Data Store and your API credentials. Ensure your `CDSAPI_URL` and `CDSAPI_KEY` are set as environment variables in your Colab environment (e.g., using `os.environ['CDSAPI_URL'] = '...'`) or configured in a `.cdsapirc` file in your home directory. If they are not found, the script will print a warning.
2.  **Data Volume**: The request is for **daily data (00:00 UTC snapshot)** for 11 different variables (including different soil layers) over 15 years (2006-2021) for the entire contiguous USA. This will still result in a **very large amount of data (potentially hundreds of gigabytes to a few terabytes)** and will take a **very long time** to download. Each variable for each year will be downloaded as a separate NetCDF (.nc) file.
3.  **Colab Limitations**: Be mindful of Colab's disk space and runtime limits. For such large downloads, you might consider running this on a cloud instance with persistent storage or downloading smaller chunks at a time.

In [11]:
print("Upgrading 'cdsapi' library...")
!pip install --upgrade cdsapi
print("'cdsapi' upgrade complete.")

Upgrading 'cdsapi' library...
'cdsapi' upgrade complete.


In [12]:
pip install "cdsapi>=0.7.7"

In [13]:
import os

# Set CDS API credentials.
# IMPORTANT: With the latest cdsapi client, the API key should *NOT* include the User ID (UID) prefix.
#            You MUST replace 'YOUR_API_KEY_HERE' with your actual Personal Access Token (PAT) from your Copernicus Data Store profile.
#            The format should now be: '906e15e9-23e2-43a2-af52-9fd4ae9da58b' (just the PAT).
os.environ['CDSAPI_URL'] = 'https://cds.climate.copernicus.eu/api'
os.environ['CDSAPI_KEY'] = '906e15e9-23e2-43a2-af52-9fd4ae9da58b'

print("CDSAPI_URL and CDSAPI_KEY environment variables have been set. Please ensure 'YOUR_API_KEY_HERE' was replaced with your actual Personal Access Token (PAT).")
print("You can now run the next cell to initialize the CDS API client and start the download.")

CDSAPI_URL and CDSAPI_KEY environment variables have been set. Please ensure 'YOUR_API_KEY_HERE' was replaced with your actual Personal Access Token (PAT).
You can now run the next cell to initialize the CDS API client and start the download.


In [14]:
import cdsapi
import os

# --- Check and setup CDS API Credentials ---
# The cdsapi client automatically looks for CDSAPI_URL and CDSAPI_KEY
# environment variables or a .cdsapirc file in the user's home directory.
# If you are setting them as environment variables in Colab, uncomment and fill in the lines below.
# Make sure to replace 'YOUR_CDSAPI_URL' and 'YOUR_CDSAPI_KEY' with your actual credentials.
# For example:
# os.environ['CDSAPI_URL'] = 'https://cds.climate.copernicus.eu/api'
# os.environ['CDSAPI_KEY'] = 'abcdefg-hijklmnop-qrstuv-wxyz' # Just the PAT, no UID prefix.

# IMPORTANT: With the latest cdsapi client, the API key format is just the Personal Access Token (PAT).
#            (e.g., 'abcdefg-hijklmnop-qrstuv-wxyz').
#            Ensure your key matches this format (no <UID>: prefix).

if 'CDSAPI_URL' not in os.environ or 'CDSAPI_KEY' not in os.environ:
    print("WARNING: CDS API credentials (CDSAPI_URL and CDSAPI_KEY) were not found as environment variables.")
    print("         Please ensure they are set, or configured in a ~/.cdsapirc file.")
    print("         The download will likely fail if credentials are not correctly set.")
    print("         If you see 'CDS API credentials found in environment variables.' below, but still get an error,")
    print("         please double-check the format of your key (just the PAT) and URL.")
else:
    print("CDS API credentials found in environment variables.")

# Initialize the CDS API client
try:
    c = cdsapi.Client()
    print("CDS API client initialized successfully.")
except AssertionError as e:
    print(f"ERROR: Failed to initialize CDS API client: {e}")
    print("Please ensure your CDSAPI_URL and CDSAPI_KEY environment variables are correctly set and formatted.")
    print("CDSAPI_URL should be 'https://cds.climate.copernicus.eu/api'")
    print("CDSAPI_KEY should be just your Personal Access Token (PAT), without the <UID>: prefix.")
    exit() # Exit to prevent further execution if client initialization fails

# --- Define Download Parameters ---
dataset_name = 'reanalysis-era5-land'

# Bounding box for the contiguous USA: [North, West, South, East]
# This covers roughly 50N, 125W, 25N, 65W.
area_usa = [50, -125, 25, -65]

years = [str(y) for y in range(2006, 2022)] # Years from 2006 to 2021 (inclusive)
months = [f'{m:02d}' for m in range(1, 13)]   # All months (01 to 12)
days = [f'{d:02d}' for d in range(1, 32)]     # All days (01 to 31)
times = ['00:00'] # Requesting a single daily snapshot at 00:00 UTC

# Map conceptual variable names to their corresponding CDS API variable names
requested_variables_map = {
    'temperature': ['2m_temperature'],
    'precipitation': ['total_precipitation'],
    'soil_moisture': [
        'volumetric_soil_water_layer_1', 'volumetric_soil_water_layer_2',
        'volumetric_soil_water_layer_3', 'volumetric_soil_water_layer_4'
    ],
    'soil_temperature': [
        'soil_temperature_level_1', 'soil_temperature_level_2',
        'soil_temperature_level_3', 'soil_temperature_level_4'
    ],
    'humidity': ['2m_dewpoint_temperature'] # 2m_dewpoint_temperature is a common proxy for surface humidity
}

# Create a directory to store the downloaded files
output_dir = 'era5_land_usa_data'
os.makedirs(output_dir, exist_ok=True)

print(f"\nStarting data download to the '{output_dir}' directory.")
print("Each variable for each year will be downloaded as a separate NetCDF file.")
print("Due to the large data volume, this process will take a significant amount of time and consume considerable disk space.")

# --- Initiate Data Download ---
for var_concept, cds_variable_list in requested_variables_map.items():
    for year in years:
        for cds_var_name in cds_variable_list:
            # Construct the filename
            filename = os.path.join(output_dir, f'{cds_var_name.replace("_", "-")}_{year}_usa.nc')

            print(f"\nRequesting '{cds_var_name}' for year {year}...")

            request_payload = {
                'variable': cds_var_name,
                'year': year,
                'month': months,
                'day': days,
                'time': times,
                'area': area_usa, # [North, West, South, East]
                'format': 'netcdf',
            }

            try:
                c.retrieve(
                    dataset_name,
                    request_payload,
                    filename
                )
                print(f"Successfully downloaded {cds_var_name} for {year} to {filename}")
            except Exception as e:
                print(f"ERROR: Failed to download {cds_var_name} for {year}: {e}")
                print("Please check your CDS API credentials and ensure the request parameters are valid.")

print("\n--- Data download process concluded ---")
print(f"Check the '{output_dir}' directory for downloaded NetCDF files. Some downloads might have failed if errors occurred.")

CDS API credentials found in environment variables.
CDS API client initialized successfully.

Starting data download to the 'era5_land_usa_data' directory.
Each variable for each year will be downloaded as a separate NetCDF file.
Due to the large data volume, this process will take a significant amount of time and consume considerable disk space.

Requesting '2m_temperature' for year 2006...


2026-05-03 07:47:19,129 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview)
INFO:ecmwf.datastores.legacy_client:[2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview)
2026-05-03 07:47:19,137

32626320948e628705ff8cc7fc952047.zip:   0%|          | 0.00/64.0M [00:00<?, ?B/s]

Successfully downloaded 2m_temperature for 2006 to era5_land_usa_data/2m-temperature_2006_usa.nc

Requesting '2m_temperature' for year 2007...


2026-05-03 07:48:46,360 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview)
INFO:ecmwf.datastores.legacy_client:[2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview)
2026-05-03 07:48:46,362

...

1810daf7c76d3561062390a74e2d0c82.zip:   0%|          | 0.00/64.1M [00:00<?, ?B/s]

Successfully downloaded 2m_dewpoint_temperature for 2020 to era5_land_usa_data/2m-dewpoint-temperature_2020_usa.nc

Requesting '2m_dewpoint_temperature' for year 2021...


2026-05-03 13:24:37,471 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview)
INFO:ecmwf.datastores.legacy_client:[2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview)
2026-05-03 13:24:37,472

7ac579105c95054f3906bf1ce4368f1c.zip:   0%|          | 0.00/64.5M [00:00<?, ?B/s]

Successfully downloaded 2m_dewpoint_temperature for 2021 to era5_land_usa_data/2m-dewpoint-temperature_2021_usa.nc

--- Data download process concluded ---
Check the 'era5_land_usa_data' directory for downloaded NetCDF files. Some downloads might have failed if errors occurred.


In [15]:
import shutil
from google.colab import files

# Create a zip archive of the directory
zip_filename = 'era5_land_usa_data.zip'
shutil.make_archive('era5_land_usa_data', 'zip', 'era5_land_usa_data')

print(f"'{zip_filename}' created. Initiating download...")

# Provide a download link
files.download(zip_filename)
print("Download initiated. Check your browser's downloads.")

'era5_land_usa_data.zip' created. Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download initiated. Check your browser's downloads.
